# 00 · Tracking de jugadores: ByteTrack vs BoT-SORT

**Objetivo:** aplicar tracking multi-objeto para asignar a cada jugador un identificador persistente, y comparar la estabilidad de ByteTrack y BoT-SORT (con re-identificación por apariencia) mediante métricas objetivas.

**Contexto:** la detección frame a frame no mantiene identidades entre frames. El tracking lo resuelve, pero en fútbol las oclusiones en aglomeraciones provocan pérdidas y reasignaciones de ID. Se mide la estabilidad con dos métricas: número de IDs únicos de jugador (menos = mejor) y duración media de track (más = mejor). El balón y los árbitros se contabilizan aparte, ya que el balón se detecta de forma intermitente e infla el conteo.

**Entrada:** vídeo de prueba y detector `best_v4.pt` (a imgsz=1280, su resolución de entrenamiento).

**Salida:** vídeo con cajas e IDs dibujados, y métricas de estabilidad.

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
detector = YOLO(str(REPO_ROOT / "models" / "best_v4.pt"))
VIDEO_IN = REPO_ROOT / "data" / "raw" / "A1606b0e6_0 (62).mp4"

print(f"Vídeo existe: {VIDEO_IN.exists()}")
print(f"Clases del detector: {detector.names}")

Vídeo existe: True
Clases del detector: {0: 'ball', 1: 'player', 2: 'referee'}


## 1. Configuración del tracker

Se crea por código el archivo de configuración de BoT-SORT con re-identificación activada (`with_reid: true`), buffer de track largo (120 frames, ~5 s) para aguantar oclusiones, y compensación de movimiento de cámara (`gmc_method: sparseOptFlow`) para los paneos de la retransmisión.

In [2]:
botsort_cfg = REPO_ROOT / "configs" / "botsort_custom.yaml"
botsort_cfg.parent.mkdir(parents=True, exist_ok=True)

contenido = """tracker_type: botsort
track_high_thresh: 0.25
track_low_thresh: 0.1
new_track_thresh: 0.6
track_buffer: 120
match_thresh: 0.8
fuse_score: true
gmc_method: sparseOptFlow
proximity_thresh: 0.5
appearance_thresh: 0.25
with_reid: true
model: auto
"""

with open(botsort_cfg, "w", encoding="utf-8") as f:
    f.write(contenido)

print(f"Config creada en: {botsort_cfg}")
print(f"Existe: {botsort_cfg.exists()}")

Config creada en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\configs\botsort_custom.yaml
Existe: True


## 2. Tracking sobre el vídeo con métricas de estabilidad

Se procesa el vídeo con `detector.track()` y `persist=True`. Se dibujan las cajas con su ID y se calculan las métricas por clase.

**Importante:** el estado del tracker se acumula entre ejecuciones dentro del mismo kernel. Para comparar configuraciones de forma justa, hay que reiniciar el kernel antes de cada prueba.

In [3]:
OUTPUT_VIDEO = REPO_ROOT / "outputs" / "tracking_validacion_botsortv2.mp4"
OUTPUT_VIDEO.parent.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(VIDEO_IN))
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps, (w, h))

# Color por clase (BGR): balón, jugador, árbitro
colores_clase = {0: (0, 255, 255), 1: (0, 0, 255), 2: (0, 0, 0)}

# Métricas de estabilidad
ids_por_clase = {0: set(), 1: set(), 2: set()}
detecciones_jugador = 0
frame_idx = 0

results = detector.track(
    source=str(VIDEO_IN),
    stream=True,
    persist=True,
    tracker=str(REPO_ROOT / "configs" / "botsort_custom.yaml"),
    conf=0.25,
    imgsz=1280,
    verbose=False
)

for rd in results:
    frame = rd.orig_img.copy()

    if rd.boxes is not None and rd.boxes.id is not None:
        boxes = rd.boxes.xyxy.cpu().numpy().astype(int)
        ids = rd.boxes.id.cpu().numpy().astype(int)
        clases = rd.boxes.cls.cpu().numpy().astype(int)

        for (x1, y1, x2, y2), tid, c in zip(boxes, ids, clases):
            ids_por_clase[c].add(tid)
            if c == 1:
                detecciones_jugador += 1
            color = colores_clase.get(c, (255, 255, 255))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID:{tid}", (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    writer.write(frame)
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"Procesados {frame_idx}/{total} frames...")

writer.release()

n_ids_jugador = len(ids_por_clase[1])
print(f"\nVídeo guardado en: {OUTPUT_VIDEO}")
print(f"IDs de JUGADOR: {n_ids_jugador}")
print(f"IDs de balón: {len(ids_por_clase[0])}, árbitro: {len(ids_por_clase[2])}")
if n_ids_jugador > 0:
    print(f"Duración media de track de jugador: {detecciones_jugador / n_ids_jugador:.0f} frames")

Procesados 50/749 frames...
Procesados 100/749 frames...
Procesados 150/749 frames...
Procesados 200/749 frames...
Procesados 250/749 frames...
Procesados 300/749 frames...
Procesados 350/749 frames...
Procesados 400/749 frames...
Procesados 450/749 frames...
Procesados 500/749 frames...
Procesados 550/749 frames...
Procesados 600/749 frames...
Procesados 650/749 frames...
Procesados 700/749 frames...

Vídeo guardado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\outputs\tracking_validacion_botsortv2.mp4
IDs de JUGADOR: 49
IDs de balón: 9, árbitro: 10
Duración media de track de jugador: 306 frames


## 3. Post-procesado: cosido de tracklets (tracklet stitching)

Cuando un jugador se pierde en una oclusión y reaparece, el tracker le asigna un ID nuevo, fragmentando su trayectoria. Como el análisis es offline, se puede post-procesar: si un track muere en el frame t y otro nace pocos frames después en una posición compatible con la trayectoria del primero (posición final + velocidad estimada), se fusionan ambos IDs.

Se aplica solo a la clase jugador, con umbrales conservadores: una fusión errónea (mezclar dos jugadores distintos) es peor que dejar el track fragmentado.

In [2]:
# Re-ejecutar el tracking guardando los datos crudos (sin escribir vídeo aún)
# REINICIA EL KERNEL antes de ejecutar esta celda, y ejecuta antes la celda 2

from collections import defaultdict

frames_dets = []                 # por frame: lista de (tid, x1, y1, x2, y2, cls)
tracks = defaultdict(list)       # tid -> lista de (frame_idx, cx_pies, cy_pies)
clase_de = {}                    # tid -> clase

results = detector.track(
    source=str(VIDEO_IN),
    stream=True,
    persist=True,
    tracker=str(REPO_ROOT / "configs" / "botsort_custom.yaml"),
    conf=0.25,
    imgsz=1280,
    verbose=False
)

frame_idx = 0
for rd in results:
    dets = []
    if rd.boxes is not None and rd.boxes.id is not None:
        boxes = rd.boxes.xyxy.cpu().numpy()
        ids = rd.boxes.id.cpu().numpy().astype(int)
        clases = rd.boxes.cls.cpu().numpy().astype(int)
        for (x1, y1, x2, y2), tid, c in zip(boxes, ids, clases):
            dets.append((tid, x1, y1, x2, y2, c))
            tracks[tid].append((frame_idx, (x1 + x2) / 2, y2))
            clase_de[tid] = c
    frames_dets.append(dets)
    frame_idx += 1
    if frame_idx % 100 == 0:
        print(f"Procesados {frame_idx} frames...")

n_jug = sum(1 for t, c in clase_de.items() if c == 1)
print(f"\nTracks guardados: {len(tracks)} totales, {n_jug} de jugador")

Procesados 100 frames...
Procesados 200 frames...
Procesados 300 frames...
Procesados 400 frames...
Procesados 500 frames...
Procesados 600 frames...
Procesados 700 frames...

Tracks guardados: 60 totales, 46 de jugador


In [12]:
# Cosido de tracklets: división por clase con histéresis + cosido con tolerancia a solapamiento
import numpy as np
from collections import defaultdict

# ---- 1. Reconstruir tracks con la clase de cada punto ----
tracks_full = defaultdict(list)
for f_idx, dets in enumerate(frames_dets):
    for (tid, x1, y1, x2, y2, c) in dets:
        tracks_full[tid].append((f_idx, (x1 + x2) / 2, y2, int(c)))

# ---- 2. Dividir tracks SOLO en cambios de clase sostenidos ----
MIN_SEG = 15

tracks2 = {}
clase2 = {}
mapa_det = {}
next_id = 1
n_divididos = 0

for tid, pts in tracks_full.items():
    segs = []
    actual = []
    pendiente = []
    c_actual = None
    for p in pts:
        c = p[3]
        if c_actual is None:
            c_actual = c
        if c == c_actual:
            if pendiente:
                actual.extend([(f, x, y, c_actual) for (f, x, y, _) in pendiente])
                pendiente = []
            actual.append(p)
        else:
            pendiente.append(p)
            if len(pendiente) >= MIN_SEG:
                if actual:
                    segs.append((c_actual, actual))
                c_actual = c
                actual = [(f, x, y, c) for (f, x, y, _) in pendiente]
                pendiente = []
    if pendiente:
        actual.extend([(f, x, y, c_actual) for (f, x, y, _) in pendiente])
    if actual:
        segs.append((c_actual, actual))

    if len(segs) > 1:
        n_divididos += 1
    for c_seg, seg in segs:
        nid = next_id; next_id += 1
        tracks2[nid] = [(f, x, y) for (f, x, y, c) in seg]
        clase2[nid] = c_seg
        for (f, x, y, c) in seg:
            mapa_det[(f, tid)] = nid

print(f"Tracks originales: {len(tracks_full)} -> tras división: {len(tracks2)}")
print(f"Tracks con cambio de clase sostenido (divididos): {n_divididos}")

# ---- 3. Cosido con tolerancia a solapamiento (captura fragmentación por swaps) ----
MAX_GAP = 75          # hueco positivo máximo (frames)
MAX_SOLAPE = 12       # solapamiento máximo tolerado (frames)
RADIO_BASE = 50
RADIO_POR_FRAME = 5
RADIO_SOLAPE = 35     # radio estricto para el caso con solapamiento

resumen = {}
pos_por_frame = {}    # nid -> dict frame -> (x, y), para consultar posición en solapes
for nid, pts in tracks2.items():
    if clase2[nid] not in (1, 2) or len(pts) < 3:
        continue
    pts_arr = np.array(pts)
    ult = pts_arr[-5:]
    if len(ult) >= 2:
        dt = ult[-1, 0] - ult[0, 0]
        vel = (ult[-1, 1:] - ult[0, 1:]) / dt if dt > 0 else np.zeros(2)
    else:
        vel = np.zeros(2)
    resumen[nid] = {
        "ini": int(pts_arr[0, 0]), "fin": int(pts_arr[-1, 0]),
        "pos_ini": pts_arr[0, 1:], "pos_fin": pts_arr[-1, 1:],
        "vel_fin": vel,
    }
    pos_por_frame[nid] = {int(f): np.array([x, y]) for f, x, y in pts}

candidatos = []
for ta, ra in resumen.items():
    for tb, rb in resumen.items():
        if ta == tb or clase2[ta] != clase2[tb]:
            continue
        gap = rb["ini"] - ra["fin"]
        if gap > MAX_GAP or gap < -MAX_SOLAPE:
            continue
        if gap > 0:
            pos_predicha = ra["pos_fin"] + ra["vel_fin"] * gap
            dist = np.linalg.norm(rb["pos_ini"] - pos_predicha)
            radio = RADIO_BASE + RADIO_POR_FRAME * gap
        else:
            # Solapamiento: comparar posición de A en el frame en que nace B
            pa = pos_por_frame[ta].get(rb["ini"])
            if pa is None:
                continue
            dist = np.linalg.norm(rb["pos_ini"] - pa)
            radio = RADIO_SOLAPE
        if dist <= radio:
            candidatos.append((dist, ta, tb))

candidatos.sort()
usados_fin, usados_ini = set(), set()
fusion = {}
for dist, ta, tb in candidatos:
    if ta in usados_fin or tb in usados_ini:
        continue
    fusion[tb] = ta
    usados_fin.add(ta)
    usados_ini.add(tb)

def raiz(t):
    while t in fusion:
        t = fusion[t]
    return t

mapa_final = {nid: raiz(nid) for nid in resumen}

n_antes_jug = sum(1 for t in resumen if clase2[t] == 1)
n_desp_jug = len(set(mapa_final[t] for t in resumen if clase2[t] == 1))
n_antes_arb = sum(1 for t in resumen if clase2[t] == 2)
n_desp_arb = len(set(mapa_final[t] for t in resumen if clase2[t] == 2))

print(f"\nFusiones realizadas: {len(fusion)}")
print(f"IDs de jugador: {n_antes_jug} -> {n_desp_jug}")
print(f"IDs de árbitro: {n_antes_arb} -> {n_desp_arb}")

Tracks originales: 60 -> tras división: 66
Tracks con cambio de clase sostenido (divididos): 3

Fusiones realizadas: 22
IDs de jugador: 43 -> 26
IDs de árbitro: 7 -> 2


In [13]:
# Re-renderizar con división por clase + cosido aplicados
OUTPUT_VIDEO3 = REPO_ROOT / "outputs" / "tracking_cosido_v5.mp4"

cap = cv2.VideoCapture(str(VIDEO_IN))
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(OUTPUT_VIDEO3), fourcc, fps, (w, h))

colores_clase = {0: (0, 255, 255), 1: (0, 0, 255), 2: (0, 0, 0)}

for frame_idx2, dets in enumerate(frames_dets):
    ret, frame = cap.read()
    if not ret:
        break
    for (tid, x1, y1, x2, y2, c) in dets:
        if c in (1, 2):
            nid = mapa_det.get((frame_idx2, tid), tid)
            tid_final = mapa_final.get(nid, nid)
        else:
            tid_final = tid
        color = colores_clase.get(c, (255, 255, 255))
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(frame, f"ID:{tid_final}", (int(x1), int(y1) - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    writer.write(frame)

cap.release()
writer.release()
print(f"Vídeo guardado en: {OUTPUT_VIDEO3}")

Vídeo guardado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\outputs\tracking_cosido_v5.mp4


In [14]:
# =====================================================================
# EXPERIMENTO DESCARTADO: corrección cinemática de swaps intra-clase
# Se conserva comentado como registro. NO ejecutar (ver Markdown anterior).
# Resultado: el único swap verificable fue un falso positivo.
# =====================================================================

# import numpy as np
# from collections import defaultdict
#
# # ---- 1. Trayectorias fusionadas por raíz (solo jugadores), suavizadas ----
# tray = defaultdict(list)
# for nid, pts in tracks2.items():
#     if clase2.get(nid) != 1 or nid not in mapa_final:
#         continue
#     tray[mapa_final[nid]].extend(pts)
#
# pos = {}
# for r, pts in tray.items():
#     pts = sorted(pts)
#     arr = np.array(pts, dtype=float)
#     fs = arr[:, 0].astype(int)
#     xy = arr[:, 1:]
#     xy_s = np.copy(xy)
#     k = 5
#     for i in range(len(xy)):
#         a, b = max(0, i - k // 2), min(len(xy), i + k // 2 + 1)
#         xy_s[i] = xy[a:b].mean(axis=0)
#     pos[r] = {int(f): p for f, p in zip(fs, xy_s)}
#
# # ---- 2. Detectar encuentros cercanos entre pares de jugadores ----
# DIST_ENC = 50   # píxeles
#
# roots = list(pos.keys())
# enc_frames = defaultdict(list)
# n_frames = len(frames_dets)
# for f in range(n_frames):
#     presentes = [(r, pos[r][f]) for r in roots if f in pos[r]]
#     for i in range(len(presentes)):
#         for j in range(i + 1, len(presentes)):
#             ra, pa = presentes[i]
#             rb, pb = presentes[j]
#             d = np.linalg.norm(pa - pb)
#             if d < DIST_ENC:
#                 enc_frames[(min(ra, rb), max(ra, rb))].append((f, d))
#
# encuentros = []
# for (a, b), lst in enc_frames.items():
#     lst.sort()
#     grupo = [lst[0]]
#     for f, d in lst[1:]:
#         if f - grupo[-1][0] <= 5:
#             grupo.append((f, d))
#         else:
#             fmin = min(grupo, key=lambda x: x[1])[0]
#             encuentros.append((fmin, a, b))
#             grupo = [(f, d)]
#     fmin = min(grupo, key=lambda x: x[1])[0]
#     encuentros.append((fmin, a, b))
# encuentros.sort()
# print(f"Encuentros cercanos detectados: {len(encuentros)}")
#
# # ---- 3. Test de hipótesis de intercambio en cada encuentro ----
# W = 12        # ventana de frames antes/después
# MIN_PTS = 6   # puntos mínimos por ventana
# DV_MIN = 1.5  # diferencia mínima de velocidades pre (px/frame)
# FACTOR = 0.45 # el intercambio debe ser claramente mejor
#
# def vel_media(r, f0, f1):
#     ps = [(f, pos[r][f]) for f in range(f0, f1 + 1) if f in pos[r]]
#     if len(ps) < MIN_PTS:
#         return None
#     fs = np.array([p[0] for p in ps], dtype=float)
#     xy = np.array([p[1] for p in ps])
#     return np.array([np.polyfit(fs, xy[:, 0], 1)[0],
#                      np.polyfit(fs, xy[:, 1], 1)[0]])
#
# swap_events = []
# for (t, a, b) in encuentros:
#     va_pre = vel_media(a, t - W, t - 1)
#     vb_pre = vel_media(b, t - W, t - 1)
#     va_post = vel_media(a, t + 1, t + W)
#     vb_post = vel_media(b, t + 1, t + W)
#     if any(v is None for v in (va_pre, vb_pre, va_post, vb_post)):
#         continue
#     if np.linalg.norm(va_pre - vb_pre) < DV_MIN:
#         continue  # velocidades indistinguibles: sin información, no se toca
#     c_no = np.linalg.norm(va_post - va_pre)**2 + np.linalg.norm(vb_post - vb_pre)**2
#     c_sw = np.linalg.norm(vb_post - va_pre)**2 + np.linalg.norm(va_post - vb_pre)**2
#     if c_sw < FACTOR * c_no:
#         swap_events.append((t, a, b))
#         print(f"SWAP en frame {t}: tracks {a} <-> {b}  (coste {c_sw:.2f}

Encuentros cercanos detectados: 81
SWAP en frame 31: tracks 25 <-> 27  (coste 1.40 vs 6.12)
SWAP en frame 582: tracks 8 <-> 9  (coste 19.53 vs 50.71)

Swaps aplicados: 2


In [15]:
# =====================================================================
# EXPERIMENTO DESCARTADO: render con corrección cinemática
# Se conserva comentado como registro. NO ejecutar.
# El vídeo final de la fase es tracking_cosido_v5.mp4 (celda 10).
# =====================================================================

# OUTPUT_VIDEO_K = REPO_ROOT / "outputs" / "tracking_cinematico.mp4"
#
# cap = cv2.VideoCapture(str(VIDEO_IN))
# fps = cap.get(cv2.CAP_PROP_FPS)
# w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# fourcc = cv2.VideoWriter_fourcc(*"mp4v")
# writer = cv2.VideoWriter(str(OUTPUT_VIDEO_K), fourcc, fps, (w, h))
#
# colores_clase = {0: (0, 255, 255), 1: (0, 0, 255), 2: (0, 0, 0)}
#
# for frame_idx2, dets in enumerate(frames_dets):
#     ret, frame = cap.read()
#     if not ret:
#         break
#     for (tid, x1, y1, x2, y2, c) in dets:
#         if c in (1, 2):
#             nid = mapa_det.get((frame_idx2, tid), tid)
#             tid_final = mapa_final.get(nid, nid)
#             if c == 1:
#                 tid_final = id_corregido(tid_final, frame_idx2)
#         else:
#             tid_final = tid
#         color = colores_clase.get(c, (255, 255, 255))
#         cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
#         cv2.putText(frame, f"ID:{tid_final}", (int(x1), int(y1) - 5),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
#     writer.write(frame)
#
# cap.release()
# writer.release()
# print(f"Vídeo guardado en: {OUTPUT_VIDEO_K}")

print("Celda de experimento descartado: no ejecuta nada (código comentado)")

Vídeo guardado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\outputs\tracking_cinematico.mp4
